# BTL02 T0205 — Ablation bộ tối ưu trên LargeCNN

Chọn GPU runtime trên Colab. Thêm `GITHUB_TOKEN` (nếu repo private) và `KAGGLE_API_TOKEN` vào Colab Secrets, cấp quyền truy cập và chạy đủ sáu cell. So sánh là giữa ba recipes (optimizer + LR + weight decay), vì các siêu tham số này được chỉ định khác nhau trong EXP-08. Toàn bộ log, checkpoint, CSV và biểu đồ được lưu ở Drive.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/BTL02/T0205')
MLRUNS_DIR = DRIVE_ROOT / 'mlruns'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'
LOGS_DIR = DRIVE_ROOT / 'logs'
FIGURES_DIR = DRIVE_ROOT / 'figures'
for folder in (MLRUNS_DIR, CHECKPOINTS_DIR, LOGS_DIR, FIGURES_DIR): folder.mkdir(parents=True, exist_ok=True)
print('Drive artifacts:', DRIVE_ROOT)


In [ ]:
import os, subprocess, tempfile
from pathlib import Path
from google.colab import userdata
REPO_URL = 'https://github.com/Duy-DS/Deep_Learning_Group.git'
BRANCH = 'exp/BTL02_T0205_ablation_optimizers'
REPO_DIR = Path('/content/Deep_Learning_Group')
token = userdata.get('GITHUB_TOKEN')
git_env = os.environ.copy()
askpass_path = None
if token:
    askpass = '#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$COLAB_GITHUB_TOKEN" ;; esac\n'
    fd, askpass_path = tempfile.mkstemp(prefix='colab-git-askpass-')
    with os.fdopen(fd, 'w', encoding='utf-8') as stream: stream.write(askpass)
    os.chmod(askpass_path, 0o700)
    git_env.update({'GIT_ASKPASS': askpass_path, 'GIT_TERMINAL_PROMPT': '0', 'COLAB_GITHUB_TOKEN': token})
try:
    if REPO_DIR.exists():
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True, env=git_env, capture_output=True, text=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True, env=git_env, capture_output=True, text=True)
    else:
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True, env=git_env, capture_output=True, text=True)
finally:
    if askpass_path and os.path.exists(askpass_path): os.remove(askpass_path)
    git_env.pop('COLAB_GITHUB_TOKEN', None)
    token = None
PROJECT_DIR = REPO_DIR / 'BTL02'
os.chdir(PROJECT_DIR)
print('Branch:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())


In [ ]:
import subprocess, sys
requirements = open('requirements.txt', encoding='utf-8').read().splitlines()
filtered = [line for line in requirements if line.strip() and not line.lstrip().startswith('#') and line.split('=', 1)[0].strip().lower() not in {'torch', 'torchvision'}]
if sys.version_info >= (3, 13):
    filtered = ['scikit-learn==1.6.1' if line.strip().lower().startswith('scikit-learn==') else line for line in filtered]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *filtered], check=True)
import torch, torchvision
print('PyTorch:', torch.__version__, '| Torchvision:', torchvision.__version__)
if not torch.cuda.is_available(): raise RuntimeError('Hãy bật GPU runtime Colab trước khi chạy.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import os, subprocess, sys
from google.colab import userdata
token = userdata.get('KAGGLE_API_TOKEN')
if not token: raise RuntimeError('Thiếu Colab Secret KAGGLE_API_TOKEN.')
os.environ['KAGGLE_API_TOKEN'] = token
del token
subprocess.run([sys.executable, '-m', 'src.data.downloader'], check=True, env=os.environ.copy())
from src.data.downloader import FER2013Downloader
split_counts = FER2013Downloader.validate_csv(PROJECT_DIR / 'data/raw/fer2013.csv')
assert split_counts == {'Training': 28709, 'PublicTest': 3589, 'PrivateTest': 3589}, split_counts
print('FER2013 validated:', split_counts)


In [ ]:
import os, subprocess, sys
os.environ['MLFLOW_TRACKING_URI'] = f'file://{MLRUNS_DIR}'
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
outputs_dir = PROJECT_DIR / 'outputs'
outputs_dir.mkdir(parents=True, exist_ok=True)
def link_drive_folder(name, target):
    link = outputs_dir / name
    if link.is_symlink(): link.unlink()
    elif link.exists():
        items = list(link.iterdir()) if link.is_dir() else []
        if link.is_dir() and all(x.name == '.gitkeep' and x.is_file() for x in items):
            for item in items: item.unlink()
            link.rmdir()
        else: raise RuntimeError(f'Không ghi đè outputs/{name} có dữ liệu.')
    link.symlink_to(target, target_is_directory=True)
link_drive_folder('checkpoints', CHECKPOINTS_DIR)
link_drive_folder('logs', LOGS_DIR)
link_drive_folder('figures', FIGURES_DIR)
configs = ['experiments/ablation/exp08_opt_adam.yaml', 'experiments/ablation/exp08_opt_adamw.yaml', 'experiments/ablation/exp08_opt_sgd_nesterov.yaml']
for config in configs:
    print(f'\n===== {config} =====', flush=True)
    subprocess.run([sys.executable, '-u', '-m', 'src.train.trainer', '--config', config], check=True, env=os.environ.copy())


In [ ]:
import os, pandas as pd, mlflow
import matplotlib.pyplot as plt
from mlflow.tracking import MlflowClient
from src.train.mlflow_logger import MLflowLogger
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
logger = MLflowLogger()
raw_path = LOGS_DIR / 't0205_all_mlflow_runs_24col.csv'
logger.export_summary_csv(str(raw_path))
runs = pd.read_csv(raw_path, encoding='utf-8-sig')
assert len(runs.columns) == 24
runs['_started'] = pd.to_datetime(runs['Start Time'], errors='coerce', utc=True)
runs = runs[(runs['params.exp_group'].astype(str) == 'EXP-08') & (runs['params.rq_tag'].astype(str) == 'Ablation_Optimizer') & (runs['params.architecture'].astype(str) == 'LargeCNN')]
latest = []
for optimizer in ('adam', 'adamw', 'sgd'):
    group = runs[runs['Name'].astype(str).str.endswith('_' + optimizer)].sort_values('_started')
    if group.empty: raise RuntimeError(f'Missing MLflow run for {optimizer}')
    row = group.iloc[-1]
    if str(row['Status']).upper() != 'FINISHED': raise RuntimeError(f'Latest {optimizer} run not FINISHED')
    latest.append(row.drop(labels=['_started']))
summary = pd.DataFrame(latest).reindex(columns=MLflowLogger.CSV_COLUMNS)
results_path = DRIVE_ROOT / 't0205_results_24col.csv'
summary.to_csv(results_path, index=False, encoding='utf-8-sig')
assert len(summary) == 3 and len(summary.columns) == 24
display(summary[['Name', 'Status', 'Run ID', 'metrics.best_epoch', 'metrics.train_loss', 'metrics.train_acc', 'metrics.best_val_acc', 'metrics.private_test_acc', 'metrics.macro_f1']])
client = MlflowClient(tracking_uri=os.environ['MLFLOW_TRACKING_URI'])
history = []
for _, run in summary.iterrows():
    for metric in ('train_loss', 'val_loss', 'train_acc', 'val_acc'):
        for point in client.get_metric_history(run['Run ID'], metric):
            if point.step > 0: history.append({'optimizer': run['params.optimizer'], 'run_id': run['Run ID'], 'metric': metric, 'epoch': point.step, 'value': point.value})
history_df = pd.DataFrame(history)
history_path = DRIVE_ROOT / 't0205_epoch_metrics.csv'
history_df.to_csv(history_path, index=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for opt in ('adam', 'adamw', 'sgd'):
    for metric, ax in [('train_loss', axes[0]), ('val_loss', axes[0]), ('train_acc', axes[1]), ('val_acc', axes[1])]:
        part = history_df[(history_df['optimizer'] == opt) & (history_df['metric'] == metric)]
        if not part.empty: ax.plot(part['epoch'], part['value'], label=f'{opt} {metric}')
axes[0].set_title('Train / Validation Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()
axes[1].set_title('Train / Validation Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend()
fig.tight_layout()
figure_path = FIGURES_DIR / 't0205_optimizer_loss_curves.png'
fig.savefig(figure_path, dpi=160, bbox_inches='tight')
print('24-col CSV:', results_path, '| epoch history:', history_path, '| curves:', figure_path)
